In [0]:
# step 1 (Creating the data )
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType

In [0]:
# step 2 Creating a list of data to clean
messy_data = [
    ("user_1", "2026-06-20", "West", "Electronics", 500.0, 25, "New York", "user1@email.com", "user_one", "2026-06-20 10:00:00", "Store_A"),
    ("user_1", "2026-06-20", "West", "Electronics", 500.0, 25, "New York", "user1@email.com", "user_one", "2026-06-20 10:00:00", "Store_A"), # Duplicate
    ("user_2", "2026-06-21", "East", "Electronics", None, 40, "Chicago", "user2@email.com", "user_two", "2026-06-21 11:00:00", "Store_B"),    # Missing price (Null)
    ("user_3", "2026-06-21", "West", "Clothing", 150.0, 22, "New York", None, "user_three", "2026-06-21 12:00:00", "Store_A"),            # Missing email (Null)
    ("user_4", "2026-06-22", "North", "Clothing", 80.0, 17, "Los Angeles", "user4@email.com", "", "2026-06-22 13:00:00", "Store_C"),         # Empty username ("")
]

columns = ["user_id", "transaction_date", "region", "product_category", "price", "age", "city", "email", "username", "raw_timestamp", "store_id"]

# Load into a Spark DataFrame
df = spark.createDataFrame(messy_data, columns)
df_sales = df 

print("Data is ready!")

Data is ready!


In [0]:
# step 3 Applying the question wise queries on the created data
#Q3
# # Drop rows where BOTH user_id and transaction_date match an existing row
df_cleaned_dupes = df.dropDuplicates(["user_id", "transaction_date"])

display(df_cleaned_dupes)

user_id,transaction_date,region,product_category,price,age,city,email,username,raw_timestamp,store_id
user_1,2026-06-20,West,Electronics,500.0,25,New York,user1@email.com,user_one,2026-06-20 10:00:00,Store_A
user_2,2026-06-21,East,Electronics,null,40,Chicago,user2@email.com,user_two,2026-06-21 11:00:00,Store_B
user_3,2026-06-21,West,Clothing,150.0,22,New York,null,user_three,2026-06-21 12:00:00,Store_A
user_4,2026-06-22,North,Clothing,80.0,17,Los Angeles,user4@email.com,,2026-06-22 13:00:00,Store_C


In [0]:
#Q4
# Keep only rows from the 'West' region
west_data = df_sales.filter(F.col("region") == "West")

# Group by category and calculate the average price
avg_sales = west_data.groupBy("product_category").agg(F.avg("price").alias("avg_sale_amount"))

display(avg_sales)

product_category,avg_sale_amount
Electronics,500.0
Clothing,150.0


In [0]:
#Q5
# # Fill any missing values in the 'username' column with the word 'Unknown'
df_filled_status = df.na.fill({"username": "Unknown"})

display(df_filled_status)

user_id,transaction_date,region,product_category,price,age,city,email,username,raw_timestamp,store_id
user_1,2026-06-20,West,Electronics,500.0,25,New York,user1@email.com,user_one,2026-06-20 10:00:00,Store_A
user_1,2026-06-20,West,Electronics,500.0,25,New York,user1@email.com,user_one,2026-06-20 10:00:00,Store_A
user_2,2026-06-21,East,Electronics,null,40,Chicago,user2@email.com,user_two,2026-06-21 11:00:00,Store_B
user_3,2026-06-21,West,Clothing,150.0,22,New York,null,user_three,2026-06-21 12:00:00,Store_A
user_4,2026-06-22,North,Clothing,80.0,17,Los Angeles,user4@email.com,,2026-06-22 13:00:00,Store_C


In [0]:
#Q6
# # Group by city, count records, and keep cities with a count greater than 100
city_counts = df.groupBy("city").count()
large_cities = city_counts.filter(F.col("count") > 100)

display(large_cities)

city,count


In [0]:
#Q8
# # Filter for age 18 to 30 AND region 'West' (representing Premium rows)
age_filter = F.col("age").between(18, 30)
premium_filter = F.col("region") == "West"

df_filtered_users = df.filter(age_filter & premium_filter)

display(df_filtered_users)

user_id,transaction_date,region,product_category,price,age,city,email,username,raw_timestamp,store_id
user_1,2026-06-20,West,Electronics,500.0,25,New York,user1@email.com,user_one,2026-06-20 10:00:00,Store_A
user_1,2026-06-20,West,Electronics,500.0,25,New York,user1@email.com,user_one,2026-06-20 10:00:00,Store_A
user_3,2026-06-21,West,Clothing,150.0,22,New York,null,user_three,2026-06-21 12:00:00,Store_A


In [0]:
# Q10
# 1. Change raw_timestamp to a real Timestamp format
df_with_time = df.withColumn("event_time", F.col("raw_timestamp").cast(TimestampType()))

# 2. Remove the old raw column
df_final_time = df_with_time.drop("raw_timestamp")

display(df_final_time)

user_id,transaction_date,region,product_category,price,age,city,email,username,store_id,event_time
user_1,2026-06-20,West,Electronics,500.0,25,New York,user1@email.com,user_one,Store_A,2026-06-20T10:00:00.000Z
user_1,2026-06-20,West,Electronics,500.0,25,New York,user1@email.com,user_one,Store_A,2026-06-20T10:00:00.000Z
user_2,2026-06-21,East,Electronics,null,40,Chicago,user2@email.com,user_two,Store_B,2026-06-21T11:00:00.000Z
user_3,2026-06-21,West,Clothing,150.0,22,New York,null,user_three,Store_A,2026-06-21T12:00:00.000Z
user_4,2026-06-22,North,Clothing,80.0,17,Los Angeles,user4@email.com,,Store_C,2026-06-22T13:00:00.000Z


In [0]:
# Q12
# # Keep rows where email IS NOT missing AND username IS NOT an empty string
good_email = F.col("email").isNotNull()
good_username = F.col("username") != ""

df_valid_rows = df.filter(good_email & good_username)

display(df_valid_rows)

user_id,transaction_date,region,product_category,price,age,city,email,username,raw_timestamp,store_id
user_1,2026-06-20,West,Electronics,500.0,25,New York,user1@email.com,user_one,2026-06-20 10:00:00,Store_A
user_1,2026-06-20,West,Electronics,500.0,25,New York,user1@email.com,user_one,2026-06-20 10:00:00,Store_A
user_2,2026-06-21,East,Electronics,null,40,Chicago,user2@email.com,user_two,2026-06-21 11:00:00,Store_B


In [0]:
#Q13
# # Calculate lowest, highest, and average prices in a single step
price_stats = df.agg(
    F.min("price").alias("min_price"),
    F.max("price").alias("max_price"),
    F.mean("price").alias("avg_price")
)

display(price_stats)

min_price,max_price,avg_price
80.0,500.0,307.5


In [0]:
# Q15     
def clean_and_calculate_revenue(input_df):
    # Step 1: Remove duplicate rows
    no_duplicates = input_df.dropDuplicates()
    # Step 2: Replace missing prices with 0
    fixed_prices = no_duplicates.na.fill({"price": 0.0})
    # Step 3: Group by store and add up total revenue
    store_revenue = fixed_prices.groupBy("store_id").agg(F.sum("price").alias("total_revenue"))
    return store_revenue
D# Run our function and display the results
pipeline_results = clean_and_calculate_revenue(df)
display(pipeline_results)

store_id,total_revenue
Store_A,650.0
Store_B,0.0
Store_C,80.0
